# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll inspect the dataset schema to find available RecordSets and their associated Fields, using their `@id` values for all references.

In [ ]:
# List available RecordSets (@id and name)
print("Available RecordSets:")
for record_set in metadata.record_sets:
    print(f"- @id: {record_set.id}, name: {record_set.name}")

# For the first RecordSet, list available fields and columns
if metadata.record_sets:
    record_set = metadata.record_sets[0]
    print(f"\nFields in RecordSet '{record_set.name}' (@id: {record_set.id}):")
    for field in record_set.fields:
        col_ids = [c.id for c in field.columns] if hasattr(field, 'columns') and field.columns else []
        print(f"  - @id: {field.id}, name: {field.name}, columns: {col_ids}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll collect all the record set @id's for extraction
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

# Load records from each RecordSet
for rs_id in record_set_ids:
    records_iter = dataset.records(record_set=rs_id)
    records = list(records_iter)
    dataframes[rs_id] = pd.DataFrame(records)

if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"Columns in main record set ({main_rs_id}):\n", dataframes[main_rs_id].columns.tolist())
    dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include outlier removal, transformations, or grouping data by key attributes to prepare it for further analysis.

Use the field `@id`s for selecting columns in the DataFrame.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

# For this dataset, we will try to use the first numeric field found.
# Let's print numeric-like fields available in the first record set.
main_df = dataframes[main_rs_id]

numeric_field_id = None
group_field_id = None

for field in metadata.record_sets[0].fields:
    # Try typical numeric field names or data types
    if hasattr(field, 'data_type') and field.data_type in ['Float', 'Number', 'Integer']:
        if field.id in main_df.columns:
            numeric_field_id = field.id
            break

# If not found, fallback: try field names
if numeric_field_id is None:
    for col in main_df.columns:
        if 'abundance' in col or 'MW' in col or 'count' in col or 'peptide' in col.lower():
            numeric_field_id = col
            break

if not numeric_field_id:
    print("No obvious numeric field found in first record set for filtering; please refer to data overview.")
else:
    print(f"Selected numeric field for analysis: {numeric_field_id}")

# Choose a group_by field (if exists)
for field in metadata.record_sets[0].fields:
    if hasattr(field, 'data_type') and field.data_type == 'Text':
        if field.id in main_df.columns and field.id != numeric_field_id:
            group_field_id = field.id
            break

# Try to use the numeric field (if it's column type compatible)
if numeric_field_id in main_df.columns:
    # Ensure data are floats
    col = numeric_field_id
    main_df[col] = pd.to_numeric(main_df[col], errors='coerce')
    threshold = main_df[col].mean() if not np.isnan(main_df[col].mean()) else 10
    filtered_df = main_df[main_df[col] > threshold]
    print(f"Filtered records with {col} > {threshold:.2f}:")
    print(filtered_df.head())
    
    norm_col = f"{col}_normalized"
    filtered_df[norm_col] = (filtered_df[col] - filtered_df[col].mean()) / filtered_df[col].std()
    print(f"\nNormalized {col} for filtered records:")
    print(filtered_df[[col, norm_col]].head())
    
    if group_field_id and group_field_id in main_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[col].mean()
        print(f"\nGrouped data by {group_field_id} (mean of {col}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll create a histogram of the selected numeric field and, if available, a boxplot grouped by the group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=20, color='skyblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    
    if group_field_id and group_field_id in main_df.columns:
        top_groups = main_df[group_field_id].value_counts().nlargest(5).index.tolist()
        plt.figure(figsize=(10,5))
        sns.boxplot(data=main_df[main_df[group_field_id].isin(top_groups)], 
                    x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id} (top 5 groups)")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded and explored the FAIR^2 dataset using the Croissant schema and the `mlcroissant` library.
- Record set and field `@id`s ensure consistency in referencing entities, as required by FAIR principles.
- Basic exploratory analysis reveals the distribution of a core numeric field and makes it easy to prepare data for downstream machine learning or bioinformatics tasks.